# Demo Pipeline — point de départ

**Objectif du challenge** (extrait de `PARTICIPANT_PACK/READ_ME.md`) : *concevoir une solution capable de
renseigner des questionnaires inconnus à partir du contexte de l'entreprise assignée, en justifiant chaque
réponse et en signalant les informations insuffisantes.*

Ce notebook montre **comment accéder aux données** du pack participant et **quel livrable produire**.
Il ne contient **aucune solution, aucune réponse et aucune méthode imposée** : à vous de concevoir votre
pipeline. Toutes les règles citées ici proviennent de `READ_ME.md`, qui reste la référence.

In [ ]:
import csv
import json
import re
from pathlib import Path

# Le pack participant se trouve à la racine du dépôt, dans PARTICIPANT_PACK/.
DATA = next(p / "PARTICIPANT_PACK" for p in [Path.cwd(), *Path.cwd().parents] if (p / "PARTICIPANT_PACK").is_dir())
print("Données :", DATA.name, "-", sorted(p.name for p in DATA.iterdir()))

## 1. Les cinq exercices

Chaque exercice associe **un questionnaire PDF** et **le contexte d'une seule entreprise** : il y a cinq
questionnaires à traiter, pas quinze combinaisons. `exercices.json` donne la correspondance.

In [ ]:
exercices = json.loads((DATA / "exercices.json").read_text(encoding="utf-8"))
for e in exercices:
    print(f"{e['exercice']}  {e['entreprise']:<28} {e['langue']:<20} {e['questionnaire']}")

## 2. Retrouver les documents d'une entreprise

D'après le READ_ME : commencez par `index_entreprises.json`, puis le `manifest.json` de l'entreprise.
Certains identifiants de source commencent par un préfixe de stockage `A/`, `B/` ou `C/` : remplacez-le par
`source_prefix_local_target` pour retrouver le fichier local. Les chemins du manifest sont relatifs au
dossier de l'entreprise.

In [ ]:
index = json.loads((DATA / "index_entreprises.json").read_text(encoding="utf-8"))


def resolve(source_ref: str) -> Path:
    """Chemin local d'un identifiant de source préfixé (règle du READ_ME)."""
    for company in index.values():
        if source_ref.startswith(company["source_prefix"]):
            return DATA / company["source_prefix_local_target"] / source_ref[len(company["source_prefix"]):]
    return DATA / source_ref


# Exemple donné par le READ_ME : A/sources/... se trouve dans entreprises/asterive_services/sources/...
example = resolve("A/sources/companies/sim-hackathon-a-client/corporate.md")
print(example.relative_to(DATA), "->", example.exists())

In [ ]:
# Manifest de l'entreprise du premier exercice
company_key = Path(exercices[0]["contexte"]).name
company_dir = DATA / index[company_key]["folder"]
manifest = json.loads((company_dir / "manifest.json").read_text(encoding="utf-8"))

print("Clés du manifest   :", list(manifest))
print("Situation au       :", manifest["as_of"])
print("Tables             :", list(manifest["table_files"]))
print("Documents sources  :", len(manifest["source_documents"]))

## 3. Lire les différents formats

Le pack contient :
- des registres **Markdown** : un en-tête rédigé, suivi d'un bloc ```` ```json ```` structuré ;
- des documents **JSON** avec une enveloppe (`document_type`, `declared_at`, `data`) ;
- des fichiers **CSV** ;
- des **tables** JSON (`tables/`) qui suivent l'organisation de la base.

Rappel du READ_ME : *un fait recopié dans plusieurs tables ne représente pas plusieurs preuves
indépendantes*, et il faut vérifier l'entité, le périmètre, la date et le statut de chaque document.

In [ ]:
def read_markdown(path: Path) -> dict:
    """Titre, texte d'en-tête et bloc JSON d'un registre Markdown."""
    text = path.read_text(encoding="utf-8")
    block = re.search(r"```json\s*\n(.*?)```", text, re.S)
    title = re.search(r"^# (.+)$", text, re.M)
    return {"title": title.group(1) if title else None,
            "header": re.sub(r"```json.*?```", "", text, flags=re.S).strip(),
            "data": json.loads(block.group(1)) if block else None}


doc = read_markdown(company_dir / manifest["source_documents"][0])
print("Titre               :", doc["title"])
print("Clés du bloc JSON   :", list(doc["data"]) if isinstance(doc["data"], dict) else type(doc["data"]).__name__)

In [ ]:
def read_json_source(path: Path) -> dict:
    return json.loads(path.read_text(encoding="utf-8"))


person_file = next((company_dir / "sources" / "persons").glob("*/*.json"))
person = read_json_source(person_file)
print("Document personne   :", person["document_type"], "- clés :", list(person["data"]))

table = read_json_source(company_dir / manifest["table_files"]["subsidiaries"])
print("Table subsidiaries  :", len(table), "lignes - colonnes :", list(table[0]))

csv_file = next((company_dir / "sources" / "companies").glob("*/*.csv"))
with csv_file.open(encoding="utf-8", newline="") as fh:
    print("CSV                 :", csv_file.name, "- colonnes :", next(csv.reader(fh)))

## 4. Les questionnaires

Les questionnaires sont des **PDF scannés** : ils ne contiennent ni couche texte ni champs de formulaire.
À vous de choisir comment en extraire les questions et les zones à remplir.

La cellule suivante utilise `pymupdf` (`pip install pymupdf`) ; elle est facultative.

In [ ]:
try:
    import pymupdf
except ImportError:
    pymupdf = None
    print("pymupdf n'est pas installé : cellule ignorée.")

if pymupdf:
    for e in exercices:
        pdf = pymupdf.open(DATA / e["questionnaire"])
        has_text = any(page.get_text().strip() for page in pdf)
        print(f"{e['exercice']}: {pdf.page_count} pages, couche texte : {'oui' if has_text else 'non'}")

In [ ]:
if pymupdf:
    from IPython.display import Image, display

    first_page = pymupdf.open(DATA / exercices[0]["questionnaire"])[0]
    display(Image(first_page.get_pixmap(dpi=50).tobytes("png")))

## 5. Le livrable attendu

Extrait de `READ_ME.md`, section *Livrables* : pour chaque exercice, **un PDF complété** et **un JSON de
réponses** avec, pour chaque champ :

- la page et le libellé du champ (le couple page + libellé permet la correspondance) ;
- la valeur ;
- l'état, parmi `answer`, `not_applicable`, `missing_information`, `bank_reserved`, `human_action` ;
- la source précise (document et section / clé / ligne) ;
- la justification ou le calcul si nécessaire.

Pour une réponse partielle, conservez les éléments connus et listez les composants inconnus.

Rappels des *Règles de réponse* : distinguez « Non », zéro, non applicable et information manquante ;
n'inventez ni donnée ni signature ; laissez les champs réservés à la banque et les signatures non exécutées.

La forme exacte du JSON ci-dessous est **une proposition** qui couvre ces éléments, pas un format imposé.

In [ ]:
STATES = {"answer", "not_applicable", "missing_information", "bank_reserved", "human_action"}


def make_answer(page: int, label: str, value, state: str, source: str | None = None,
                justification: str | None = None, missing: list[str] | None = None) -> dict:
    return {"page": page, "label": label, "value": value, "state": state, "source": source,
            "justification": justification, "missing": missing or []}


# Enregistrements FACTICES : ils montrent la forme, pas le contenu d'un vrai questionnaire.
example_answers = [
    make_answer(1, "<libellé du champ>", "<valeur>", "answer",
                source="<fichier>#<section, clé ou ligne>", justification="<pourquoi cette valeur>"),
    make_answer(1, "<autre libellé>", None, "missing_information",
                source="<fichier>#<clé>", justification="<pourquoi l'information manque>",
                missing=["<composant inconnu>"]),
]
print(json.dumps(example_answers, ensure_ascii=False, indent=2))

In [ ]:
def check_format(answers: list[dict]) -> list[str]:
    """Contrôle de FORME uniquement (champs présents, état autorisé) : il ne juge pas les réponses."""
    problems = []
    for i, a in enumerate(answers):
        for key in ("page", "label", "value", "state", "source"):
            if key not in a:
                problems.append(f"#{i}: champ '{key}' absent")
        if a.get("state") not in STATES:
            problems.append(f"#{i}: état inconnu {a.get('state')!r}")
        if a.get("state") == "missing_information" and not a.get("missing"):
            problems.append(f"#{i}: préciser les composants manquants")
    return problems


print(check_format(example_answers) or "Format OK")

# Pour enregistrer votre propre livrable, par exemple :
# out = Path("submission") / "form_01.answers.json"
# out.parent.mkdir(exist_ok=True)
# out.write_text(json.dumps(my_answers, ensure_ascii=False, indent=2), encoding="utf-8")

## 6. Exemple : notre pipeline appliqué à form_01

Cette section applique **notre pipeline de référence** au premier exercice (*form_01*, « Fiche
connaissance client » d'**Asterive Services SAS**), sur les vraies données du pack. C'est **un exemple de
démarche à suivre**, pas le corrigé officiel : vos réponses doivent venir de votre propre solution.

Le moteur de réponse est reproduit ici en code autonome (bibliothèque standard uniquement) :

```
documents ─► ingestion + fiabilité ─► faits avec preuve ─► schéma des champs ─► règles ─► validation ─► JSON
```

Il est **déterministe** (aucun LLM) et chaque réponse porte sa **preuve** (document + pointeur JSON ou
phrase). Les étapes de lecture du PDF scanné (OCR, localisation des zones) et de rendu du PDF complété ne
sont pas reproduites dans ce notebook.

In [ ]:
# 1) Ingestion : charger les documents de l'entreprise de form_01 et qualifier leur fiabilité.
FORM = exercices[0]                                   # form_01
cdir = DATA / FORM["contexte"]
mf = json.loads((cdir / "manifest.json").read_text(encoding="utf-8"))

SUBJECT_RECORDS = {"personal_facts", "entity_facts", "entity_registry_extract", "person_relationships"}
CONTEXTUAL = {"invoice", "hr_plan"}                   # enregistrements sans valeur de preuve KYC
ADMISSIBLE = {"primaire", "fiche sujet", "dérivé"}    # rôles pouvant justifier une réponse


def load_document(rel: str) -> dict:
    path = cdir / rel
    doc = {"path": rel, "subject": path.parent.name, "prose": ""}
    if path.suffix == ".md":                          # registre : en-tête rédigé + bloc JSON
        parsed = read_markdown(path)
        return doc | {"type": path.stem, "data": parsed["data"], "prose": parsed["header"], "role": "primaire"}
    if path.suffix == ".csv":                         # tableau dérivé du registre financier
        with path.open(encoding="utf-8", newline="") as fh:
            return doc | {"type": path.stem, "data": list(csv.DictReader(fh)), "role": "dérivé"}
    env = read_json_source(path)
    data, role = env["data"], "fiche sujet" if env["document_type"] in SUBJECT_RECORDS else "primaire"
    if path.stem.endswith("_facts") and path.with_name(path.stem.removesuffix("_facts") + ".md").exists():
        role = "dérivé"                               # copie du bloc JSON d'un .md : pas une preuve indépendante
    if isinstance(data, dict) and data.get("record_status") == "superseded":
        role = "historique"                           # remplacé par un document plus récent
    if isinstance(data, dict) and data.get("record_type") in CONTEXTUAL:
        role = "contexte"
    return doc | {"type": env["document_type"], "data": data, "role": role}


documents = [load_document(rel) for rel in mf["source_documents"]]
for d in documents:
    print(f"{d['role']:<12} {d['path']}")

In [ ]:
# 2) Faits : chaque valeur d'un document admissible devient un fait (document, pointeur JSON) -> valeur.
def walk(node, ptr=""):
    yield ptr, node
    if isinstance(node, dict):
        for k, v in node.items():
            yield from walk(v, f"{ptr}/{k}")
    elif isinstance(node, list):
        for i, v in enumerate(node):
            yield from walk(v, f"{ptr}/{i}")


facts = {(d["path"], p): v for d in documents if d["role"] in ADMISSIBLE for p, v in walk(d["data"]) if p}
docs = {(d["type"], d["subject"]): d for d in documents if d["role"] in ADMISSIBLE}

CLIENT = mf["client_subsidiary_id"]
subsidiaries = read_json_source(cdir / mf["table_files"]["subsidiaries"])
PARENT = next((s["subsidiary_id"] for s in subsidiaries
               if s["perimeter_tags"] == "upstream_parent_excluded_from_reporting"), None)
SCOPE = {s["subsidiary_name"].casefold() for s in subsidiaries        # client + descendants contrôlés,
         if s["perimeter_tags"] in ("reporting_client", "controlled_descendant")}   # sans la maison mère


def find(doc_type, pointer, subject):
    """Fait trouvé dans le document (type, sujet) avec sa preuve, ou None."""
    d = docs.get((doc_type, subject))
    if d is None or (d["path"], pointer) not in facts:
        return None
    value = facts[(d["path"], pointer)]
    return {"source": d["path"], "pointer": pointer, "value": value,
            "excerpt": value if isinstance(value, str) else json.dumps(value, ensure_ascii=False)}


def sentence(doc_type, needle, subject):
    """Phrase de l'en-tête rédigé d'un registre (les règles d'interprétation y sont écrites)."""
    d = docs.get((doc_type, subject))
    for s in re.split(r"(?<=[.!?])\s+", d["prose"] if d else ""):
        if needle.casefold() in s.casefold():
            return {"source": d["path"], "pointer": "(en-tête)", "value": None, "excerpt": s.strip()}


print(len(facts), "faits ; client :", CLIENT, "; maison mère (hors périmètre) :", PARENT)

In [ ]:
# 3) Schéma de form_01 : chaque champ dit quelle notion il demande, jamais sa réponse.
COUNTRIES = ["Corée du nord", "Crimée", "Cuba", "Irak", "Iran", "Myanmar", "Russie", "Soudan", "Sud-Soudan",
             "Syrie", "Venezuela"]
schema = [(1, "Dénomination sociale", "legal_name"), (1, "Code SIREN / n° d'enregistrement", "registration"),
          (1, "Société cotée", "listed"), (1, "Marché de cotation", "market"),
          (1, "Nom de la maison mère", "parent.name"), (1, "Pays d'immatriculation", "parent.incorporation"),
          (1, "Pays de résidence fiscale", "parent.tax_residence"), (1, "Adresse de la maison mère", "parent.address")]
schema += [(1 if i < 9 else 2, c, "country") for i, c in enumerate(COUNTRIES)]
schema += [(2, "Représenté par", "signatory.name"), (2, "En qualité de", "signatory.role"),
           (2, "Signé le", "signature.date"), (2, "Signature", "signature")]

# Où chercher chaque notion : toutes les sources indépendantes, pour croiser les preuves.
C, P = CLIENT, PARENT
SOURCES = {
    "legal_name": [("corporate", "/name", C), ("mandate", "/name", C)],
    "registration": [("corporate", "/registration", C)],
    "listed": [("corporate", "/listed", C)],
    "market": [("corporate", "/market", C)],
    "parent.name": [("corporate", "/parent/name", C), ("ownership", "/parent/name", C),
                    ("entity_facts", "/subsidiary_name", P)],
    "parent.incorporation": [("corporate", "/parent/incorporation", C), ("ownership", "/parent/incorporation", C),
                             ("entity_registry_extract", "/incorporation_country", P)],
    # L'extrait de registre de la maison mère précise qu'il n'établit pas de résidence fiscale : pas une source ici.
    "parent.tax_residence": [("corporate", "/parent/tax_residence", C), ("ownership", "/parent/tax_residence", C)],
    "parent.address": [("corporate", "/parent/address", C), ("ownership", "/parent/address", C),
                       ("entity_registry_extract", "/registered_office", P), ("entity_facts", "/address", P)],
    "signatory.name": [("mandate", "/signer/name", C)],
    "signatory.role": [("mandate", "/signer_role", C)],
}
print(len(schema), "champs")

In [ ]:
# 4) Règles : conditions, croisement des sources, absence documentée, déclaration négative, signature.
COUNTRY_EN = {"Corée du nord": "North Korea", "Crimée": "Crimea", "Irak": "Iraq", "Russie": "Russia",
              "Soudan": "Sudan", "Sud-Soudan": "South Sudan", "Syrie": "Syria", "FR": "France"}


def norm(value):
    return COUNTRY_EN.get(value, value).casefold() if isinstance(value, str) else value


def result(state, value, evidence, reason, missing=None):
    return {"state": state, "value": value, "evidence": [e for e in evidence if e], "reason": reason,
            "missing": missing or []}


def lookup(concept, label):
    found = [f for f in (find(*s) for s in SOURCES[concept]) if f]
    known = [f for f in found if f["value"] not in (None, "")]
    if known:
        if len({norm(f["value"]) for f in known}) > 1:          # sources en désaccord : on ne choisit pas
            return result("missing_information", None, known, "Sources contradictoires.", [label])
        return result("answer", known[0]["value"], known, f"{len(known)} source(s) concordante(s).")
    notes = [find(doc, ptr + "_note", subj) for doc, ptr, subj in SOURCES[concept]]
    notes = [n for n in notes if n and n["value"]]
    if notes:                                                   # l'absence est expliquée par la source
        return result("missing_information", None, found + notes[:1], "Information non fournie : " + notes[0]["value"],
                      [label])
    return result("missing_information", None, found, "Valeur absente des sources.", [label])


def country_activity(label):
    country = COUNTRY_EN.get(label, label)
    acts, perimeter = find("activities", "/activities", C), find("corporate", "/perimeter", C)
    current, planned = [], []
    for i, e in enumerate(acts["value"] if acts else []):
        if e.get("country") == country and (not e.get("entity") or e["entity"].casefold() in SCOPE):
            (current if e.get("current") else planned).append(find("activities", f"/activities/{i}", C))
    for i, e in enumerate(perimeter["value"] if perimeter else []):
        if e.get("country") == country and e["name"].casefold() in SCOPE:     # implantation dans le pays
            current.append(find("corporate", f"/perimeter/{i}", C))
    if current:
        return result("answer", "Oui", current, "Activité ou implantation actuelle dans le périmètre déclarant.")
    if planned:
        return result("answer", "Envisagée", planned, "Activité seulement envisagée.")
    negative = find("activities", "/negative_declaration", C)
    if negative and negative["value"]:
        evidence = [acts, negative, sentence("finance", "Unlisted jurisdictions", C),
                    sentence("corporate", "reporting group consists", C)]
        return result("answer", "Non", evidence, "Absent du registre d'activités ; déclaration négative complète.")
    return result("missing_information", None, [], "Aucune source ne couvre ce pays.", [label])


def resolve(label, concept):
    if concept == "market":                                     # « Marché de cotation » : seulement si cotée
        listed = find("corporate", "/listed", C)
        if not (listed and listed["value"] is True):
            return result("not_applicable", None, [listed], "Champ conditionnel : la société n'est pas cotée.")
    if concept.startswith("parent."):                           # « Votre maison mère (si filiale) »
        parent = find("corporate", "/parent", C)
        if not (parent and parent["value"]):
            return result("not_applicable", None, [parent], "Aucune maison mère déclarée.")
    if concept == "country":
        return country_activity(label)
    if concept in ("signature", "signature.date"):              # jamais de signature ni de date inventée
        evidence = [sentence("mandate", "No signature is supplied", C),
                    sentence("mandate", "not an actual executed signature", C), find("mandate", "/authority", C)]
        if concept == "signature":
            return result("human_action", None, evidence, "Aucune signature exécutée fournie : signature humaine requise.")
        date = find("mandate", "/date", C)
        return result("human_action", None, [date, *evidence],
                      f"Date de signature inconnue ; date de complétion de l'exercice : {date['value'] if date else '?'}.")
    r = lookup(concept, label)
    if concept == "listed" and r["state"] == "answer":
        r["value"] = "Oui" if r["value"] else "Non"             # « Non » est une réponse, pas une absence
    return r


results = [(page, label, resolve(label, concept)) for page, label, concept in schema]
rows = ["| page | champ | état | valeur |", "|---|---|---|---|"]
rows += [f"| {p} | {l} | `{r['state']}` | {r['value'] if r['value'] is not None else ''} |" for p, l, r in results]

from IPython.display import Markdown, display
display(Markdown("\n".join(rows)))

In [ ]:
# 5) Validation : chaque preuve est relue dans le fichier source ; une réponse non prouvée n'est pas gardée.
def still_true(e) -> bool:
    if e["pointer"] == "(en-tête)":
        return e["excerpt"] in next(d for d in documents if d["path"] == e["source"])["prose"]
    node = load_document(e["source"])["data"]
    for token in e["pointer"].strip("/").split("/"):
        node = node[int(token)] if isinstance(node, list) else node.get(token)
    return node == e["value"]


for page, label, r in results:
    if r["state"] == "answer" and not (r["evidence"] and all(still_true(e) for e in r["evidence"])):
        r.update(state="missing_information", value=None, missing=[label], reason="Réponse non prouvée.")
print("preuves vérifiées :", sum(len(r["evidence"]) for _, _, r in results),
      "| réponses sans preuve :", sum(1 for _, _, r in results if r["state"] == "answer" and not r["evidence"]))

In [ ]:
# 6) Livrable : JSON au format du READ_ME (page/libellé, valeur, état, source précise, justification).
form01_answers = [
    make_answer(page, label, r["value"], r["state"],
                source="; ".join(f"{e['source']}#{e['pointer']}" for e in r["evidence"]) or None,
                justification=r["reason"], missing=r["missing"])
    for page, label, r in results
]
print(check_format(form01_answers) or "Format OK")
print({s: sum(1 for a in form01_answers if a["state"] == s) for s in sorted(STATES)})

# Une réponse en détail : la résidence fiscale de la maison mère n'est PAS déduite de son pays d'immatriculation.
print(json.dumps(next(a for a in form01_answers if a["label"] == "Pays de résidence fiscale"),
                 ensure_ascii=False, indent=2))

**À retenir de cet exemple**

- Les documents ne se valent pas : l'avis d'adresse remplacé, la facture et le plan de formation sont lus
  mais ne servent jamais de preuve ; les copies `*_facts.json` ne comptent pas comme des preuves de plus.
- Une valeur absente n'est pas devinée : la résidence fiscale de la maison mère reste `missing_information`,
  avec la phrase de la source qui l'explique.
- « Non » est une réponse, une question conditionnelle non déclenchée est `not_applicable`, et la signature
  comme sa date restent une `human_action`.
- Chaque réponse est relue dans sa source avant d'être écrite dans le livrable.

À vous d'appliquer — ou d'améliorer — cette démarche aux quatre autres questionnaires, puis de produire les
PDF complétés.

## 7. À vous de jouer

À construire librement, avec les outils de votre choix :

1. extraire les questions et les zones de chaque questionnaire ;
2. retrouver dans les sources de l'entreprise assignée les éléments qui y répondent ;
3. décider de la valeur et de l'état de chaque champ, avec sa source et sa justification ;
4. produire le JSON de réponses et le PDF complété.

À remettre aussi (READ_ME, *Livrables*) : votre code et un README permettant de reproduire le résultat
(installation, commande, modèles et dépendances, durée et coûts observés). Le canal de remise, l'horaire,
les outils autorisés et le budget API seront communiqués par les organisateurs.

*Exécution de ce notebook : `pip install jupyter pymupdf`, puis `jupyter lab notebooks/demo_pipeline.ipynb`
depuis la racine du dépôt.*